# When two moves compose

One commuting square takes integers to parity. A second takes parity to a single token: even and odd are forgotten too. Each square is honest on its own. The composite should still commute: look after both moves, or move the look twice.

Loss does not reset at the join. If the first move records an erasure and the second records an approximation, the composite records both, and the masses add.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_mur_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "mur" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "mur"
elif (p / "mur" / "code").is_dir():
    root = p / "mur"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
repo_root = root.parents[2] if len(root.parents) >= 3 else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from sympy import Rational
from coda_mur_core import (
    DescentSquareCertificate,
    LossLedger,
    MorphismKind,
    paste_certificates,
)

## Two commuting squares

$F$ quotients integers to parity and records an erasure of mass $1/5$. $G$ approximates parity by a single token and records an approximation of mass $2/5$. They meet at `parity`: the target of $F$ is the source of $G$.

In [2]:
chi_f = DescentSquareCertificate(
    source_regime="integers",
    target_regime="parity",
    morphism_name="F_mod2",
    morphism_kind=MorphismKind.QUOTIENT,
    chi_certified=True,
    commutes=True,
    loss_ledger=LossLedger(e=True, loss_mass=Rational(1, 5)),
    carrier=("2", "3", "4"),
    state_map=lambda x: str(int(x) % 2),
    observation_map=lambda x: str(int(x) % 2),
    source_look=lambda x: x,
    target_look=lambda x: x,
)
chi_g = DescentSquareCertificate(
    source_regime="parity",
    target_regime="one_token",
    morphism_name="G_collapse",
    morphism_kind=MorphismKind.APPROXIMATION,
    chi_certified=True,
    commutes=True,
    loss_ledger=LossLedger(a=True, loss_mass=Rational(2, 5)),
    carrier=("0", "1"),
    state_map=lambda x: "*",
    observation_map=lambda x: "*",
    source_look=lambda x: x,
    target_look=lambda x: x,
)

pasted = paste_certificates(chi_f, chi_g)
print("source:", pasted.source_regime)
print("target:", pasted.target_regime)
print("kind:", pasted.morphism_kind.value)
print("still commutes?", pasted.commutes)
print("erasure bit:", pasted.loss_ledger.e, "approximation bit:", pasted.loss_ledger.a)
print("loss mass:", pasted.loss_ledger.loss_mass)
assert pasted.source_regime == "integers"
assert pasted.target_regime == "one_token"
assert pasted.morphism_kind == MorphismKind.APPROXIMATION
assert pasted.commutes is True
assert pasted.loss_ledger.e is True and pasted.loss_ledger.a is True
assert pasted.loss_ledger.loss_mass == Rational(3, 5)

source: integers
target: one_token
kind: approximation
still commutes? True
erasure bit: True approximation bit: True
loss mass: 3/5


## Loss adds, and addition associates

A third move after the token, with mass $1/5$, should not care whether you added left-to-right or right-to-left. The empty ledger is a unit: adding it changes nothing.

In [3]:
l1 = LossLedger(e=True, loss_mass=Rational(1, 5))
l2 = LossLedger(a=True, loss_mass=Rational(2, 5))
l3 = LossLedger(i=True, loss_mass=Rational(1, 5))
empty = LossLedger.empty()

left = (l1.accumulate(l2)).accumulate(l3)
right = l1.accumulate(l2.accumulate(l3))
print("left-associated mass:", left.loss_mass)
print("right-associated mass:", right.loss_mass)
assert left == right
assert left.loss_mass == Rational(4, 5)
assert empty.accumulate(l1) == l1
assert l1.accumulate(empty) == l1
print("empty ledger is a unit")

left-associated mass: 4/5
right-associated mass: 4/5
empty ledger is a unit


## If the pictures do not meet, they do not paste

A square that starts at `other_picture` cannot follow a square that ended at `parity`. The calculator raises before building a composite. Two moves compose only when the intermediate picture is the same picture.

In [4]:
chi_bad = DescentSquareCertificate(
    source_regime="other_picture",
    target_regime="one_token",
    morphism_name="G_mismatch",
    morphism_kind=MorphismKind.EXACT,
    chi_certified=True,
    commutes=True,
    carrier=("2", "3", "4"),
    state_map=lambda x: x,
    observation_map=lambda x: x,
    source_look=lambda x: x,
    target_look=lambda x: x,
)
try:
    paste_certificates(chi_f, chi_bad)
    raise AssertionError("mismatch should refuse")
except ValueError as err:
    print("refused:", err)
    assert "Cannot paste certificates" in str(err)

refused: Cannot paste certificates: target 'parity' does not match source 'other_picture'
